# Process to be used by frontend code

Initial iteration to plot clusters on UI from Google Colab outputs

In [73]:
import json
from collections import defaultdict

### Get individual artist breakdowns

In [74]:
def process_file(data: list, source_label: str):
    result       = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: {"keywords": [], "points": []})))
    artist_names = {}

    for n_entry in data:
        n = str(n_entry["n"])
        for cluster in n_entry["clusters"]:
            cluster_key = str(cluster["key"])
            keywords    = cluster.get("keywords", [])
            for point in cluster["data"]:
                artist_id = point.get("artist_id")
                if artist_id is None:
                    continue

                artist_id_str = str(artist_id)

                if point.get("artist_name"):
                    artist_names[artist_id_str] = point["artist_name"]

                bucket = result[artist_id_str][n][cluster_key]
                if not bucket["keywords"]:
                    bucket["keywords"] = keywords

                bucket["points"].append({
                    "text":               point.get("text"),
                    "distance_to_center": point.get("distance_to_center"),
                    "is_artwork":         point.get("is_artwork"),
                    "source_idx":         point.get("source_idx"),
                    "institution_id":     point.get("institution_id"),
                    "source_actor":        source_label
                })

    return dict(result), artist_names


def merge_results(artist_result, institution_result):
    for artist_id, ns in institution_result.items():
        for n, clusters in ns.items():
            for cluster_key, payload in clusters.items():
                target = (artist_result
                          .setdefault(artist_id, {})
                          .setdefault(n, {})
                          .setdefault(cluster_key, {"keywords": [], "points": []}))
                if not target["keywords"] and payload["keywords"]:
                    target["keywords"] = payload["keywords"]
                target["points"].extend(payload["points"])
    return artist_result

def build_output(merged: dict) -> list:
    output = []

    for artist_id_str, ns in merged.items():
        n_values = []
        for n, clusters in sorted(ns.items(), key=lambda x: int(x[0])):
            cluster_list = []
            for ck, cv in sorted(clusters.items(), key=lambda x: int(x[0])):
                cluster_list.append({
                    "cluster": int(ck),
                    "points": sorted(
                        cv["points"],
                        key=lambda p: (p.get("source_actor", ""), p.get("source_idx") or "")
                    )
                })
            n_values.append({
                "n": int(n),
                "clusters": cluster_list
            })

        output.append({
            "artist_id": int(artist_id_str),
            "n_values":  n_values,
        })

    output.sort(key=lambda x: x["artist_id"])
    return output

In [75]:
with open("../../data/colab/ar_clusters.json", encoding="utf-8") as f: artist_data = json.load(f)
with open("../../data/colab/inst_clusters.json", encoding="utf-8") as f: institution_data = json.load(f)

In [76]:
artist_result, artist_names_a = process_file(artist_data, source_label="artist")
institution_result, artist_names_i = process_file(institution_data, source_label="institution")

In [77]:
merged = merge_results(artist_result, institution_result)
all_artist_names = {**artist_names_i, **artist_names_a}
output = build_output(merged)

print(f"Total artists in output: {len(output)}")

Total artists in output: 30


In [78]:
with open("../../data/ui/artist_points.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

total_points = sum(
    len(c["points"])
    for artist in output
    for nv in artist["n_values"]
    for c in nv["clusters"]
)

In [79]:
def build_flat_from_source(data: list, source_label: str) -> dict:
    points = {}

    for n_entry in data:
        for cluster in n_entry["clusters"]:
            top_keyword = cluster["keywords"][0] if cluster["keywords"] else None
            for point in cluster["data"]:
                source_idx = point.get("source_idx")
                artist_id  = point.get("artist_id")
                text       = point.get("text")
                if not source_idx or artist_id is None:
                    continue

                key = (source_idx, text)

                if key not in points:                    points[key] = {
                        "artist_id":      artist_id,
                        "text":           text,
                        "is_artwork":     point.get("is_artwork"),
                        "institution_id": point.get("institution_id"),
                        "source_actor":    source_label,
                        "keywords":       set(),
                    }

                if top_keyword:
                    points[key]["keywords"].add(top_keyword)

    return points

In [80]:
artist_points_raw      = build_flat_from_source(artist_data,      source_label="artist")
institution_points_raw = build_flat_from_source(institution_data, source_label="institution")

all_points = dict(artist_points_raw)
for key, point in institution_points_raw.items():
    if key in all_points:
        all_points[key]["keywords"].update(point["keywords"])
    else:
        all_points[key] = point

for p in all_points.values():
    p["keywords"] = sorted(p["keywords"])

sorted_points = sorted(all_points.items(), key=lambda x: (x[1]["artist_id"], x[0]))

artist_grouped = [
    {
        "artist_id": artist_id,
        "points": [
            {"source_idx": key[0], **{k: v for k, v in p.items() if k != "artist_id"}}
            for key, p in points
        ]
    }
    for artist_id, points in groupby(sorted_points, key=lambda x: x[1]["artist_id"])
]

with open("../../data/ui/artist_points_flat.json", "w", encoding="utf-8") as f:
    json.dump(artist_grouped, f, indent=2)

print(f"Written {len(artist_grouped)} artists, {len(all_points)} unique points")

Written 30 artists, 1798 unique points
